<a href="https://colab.research.google.com/github/Teja3993/Soft_Computing_Lab_exercises/blob/main/Linear_Regression_Neural_Network_25th_March_Soft_Computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np

# 1. Training Data (Your original 10 points)
X_train = np.array([
    [1.50, 45], [1.55, 50], [1.60, 55], [1.65, 60], [1.70, 65],
    [1.75, 70], [1.60, 70], [1.65, 75], [1.70, 80], [1.75, 85]
])
y_train = np.array([
    20.0, 20.8, 21.5, 22.0, 22.5, 22.9, 27.3, 27.5, 27.7, 27.8
]).reshape(-1, 1)

# 2. Validation Data (New, unseen people to test true accuracy)
X_val = np.array([
    [1.58, 58],  # Actual BMI ~ 23.2
    [1.68, 72],  # Actual BMI ~ 25.5
    [1.80, 90]   # Actual BMI ~ 27.8
])
y_val = np.array([23.2, 25.5, 27.8]).reshape(-1, 1)

# 3. Normalize input (VERY IMPORTANT: Use training stats to scale both datasets)
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

X_train_scaled = (X_train - train_mean) / train_std
X_val_scaled = (X_val - train_mean) / train_std

# Network architecture
input_size = 2
hidden1_size = 5
hidden2_size = 4
output_size = 1

# Initialize weights
np.random.seed(42)
W1 = np.random.randn(input_size, hidden1_size)
b1 = np.zeros((1, hidden1_size))
W2 = np.random.randn(hidden1_size, hidden2_size)
b2 = np.zeros((1, hidden2_size))
W3 = np.random.randn(hidden2_size, output_size)
b3 = np.zeros((1, output_size))

# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

# Training parameters
lr = 0.01
epochs = 500

train_losses = []
val_losses = []

# Training loop
for epoch in range(epochs):

    # ---- Training Forward Pass ----
    Z1 = np.dot(X_train_scaled, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)
    Z3 = np.dot(A2, W3) + b3
    y_hat_train = Z3

    # ---- Validation Forward Pass (NO BACKPROPAGATION HERE) ----
    Z1_val = np.dot(X_val_scaled, W1) + b1
    A1_val = relu(Z1_val)
    Z2_val = np.dot(A1_val, W2) + b2
    A2_val = relu(Z2_val)
    Z3_val = np.dot(A2_val, W3) + b3
    y_hat_val = Z3_val

    # ---- Calculate Losses ----
    train_loss = np.mean((y_hat_train - y_train) ** 2)
    val_loss = np.mean((y_hat_val - y_val) ** 2)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # ---- Backpropagation (Only on Training Data) ----
    dZ3 = (y_hat_train - y_train)
    dW3 = np.dot(A2.T, dZ3) / len(X_train)
    db3 = np.sum(dZ3, axis=0, keepdims=True) / len(X_train)

    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / len(X_train)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / len(X_train)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X_train_scaled.T, dZ1) / len(X_train)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / len(X_train)

    # ---- Update Weights ----
    W3 -= lr * dW3
    b3 -= lr * db3
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if epoch % 50 == 0:
        print(f"Epoch {epoch:<4} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# ---- Final Predictions on Validation Set ----
print("\n--- Final Validation Predictions ---")
for i in range(len(X_val)):
    print(f"Features: H={X_val[i][0]}m, W={X_val[i][1]}kg | Predicted BMI: {y_hat_val[i][0]:.2f} --> Actual BMI: {y_val[i][0]}")

Epoch 0    | Train Loss: 586.6529 | Val Loss: 653.7767
Epoch 50   | Train Loss: 0.0459 | Val Loss: 0.4038
Epoch 100  | Train Loss: 0.0193 | Val Loss: 0.2841
Epoch 150  | Train Loss: 0.0123 | Val Loss: 0.2294
Epoch 200  | Train Loss: 0.0093 | Val Loss: 0.1993
Epoch 250  | Train Loss: 0.0078 | Val Loss: 0.1812
Epoch 300  | Train Loss: 0.0069 | Val Loss: 0.1698
Epoch 350  | Train Loss: 0.0064 | Val Loss: 0.1620
Epoch 400  | Train Loss: 0.0060 | Val Loss: 0.1564
Epoch 450  | Train Loss: 0.0057 | Val Loss: 0.1520

--- Final Validation Predictions ---
Features: H=1.58m, W=58.0kg | Predicted BMI: 23.07 --> Actual BMI: 23.2
Features: H=1.68m, W=72.0kg | Predicted BMI: 26.13 --> Actual BMI: 25.5
Features: H=1.8m, W=90.0kg | Predicted BMI: 27.63 --> Actual BMI: 27.8
